<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/gaussienne/gaus_double_diag_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-experts MNIST — Double diagonalisation L² → Sobolev

Experts gaussiens anisotropes uniquement.

Pour chaque expert :
1. Dictionnaire de Gaussiennes anisotropes (centres + σ) — **inchangé**
2. Gram L² `G0 = (1/n) AᵀA` → haut du spectre (`n_var` / seuils relatifs)
3. Gram Sobolev `G_H` (mécanisme Monte-Carlo actuel) → projection `H_K = Vᵀ G_H V` → bas du spectre (`n_sob` / seuils)
4. Ridge classique dans la base des fonctions sélectionnées

Pas de Wendland, miso, Phase 2, ni sélection gloutonne.

Hyperparamètres de sélection spectrale indépendants :
- `params_L2` : max_n_var, mu_ceil, mu_floor (relatifs à μ_max)
- `params_Sob` : max_n_sob, lam_ceil, lam_floor, use_relative


In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score
from sklearn.svm import SVC
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV
import time

# ── HELPERS classification ────────────────────────────────────────────────
def cls_acc(f, y):
    sg = np.sign(f)
    return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))

def cls_auc(f, y):
    try:
        return float(roc_auc_score((y > 0).astype(int), f))
    except Exception:
        return float('nan')

def cls_str(f, y):
    return f"acc={cls_acc(f,y):.4f} AUC={cls_auc(f,y):.4f}"


# ══════════════════════════════════════════════════════════════════════════════
# HYPERPARAMÈTRES
# ══════════════════════════════════════════════════════════════════════════════
params_shared = {
    "neg_digits": [3, 5], "pos_digits": [8], "img_size": 7,
    "seed": 7787225,
    "n_train": 300, "n_test": 5000,
    "n_dirs": 50, "batch_dirs": 10,
    "deg_P": 3,   # degré du Ridge polynomial de comparaison
}
params_shared["n_unlabeled"] = np.maximum(6000 - params_shared["n_train"], 500)
params_shared["train_center_ratio"] = 0.5 + params_shared["n_train"] / (2 * 4000)

params_gauss = {
    "weights": {0: 0, 1: 1, 2: 0.1, 3: 0.01},
    "lambda_reg": 0.0001,          # utilisé uniquement pour le Ridge final sur les w_i
    "thresh_factor": 1e-5,      # (legacy, peu utilisé ici)
    "sigma_min": 0.7, "sigma_max": 5,
    "n_dict": 2000, "n_centres": 800,
    "n_G": 1000,
    "n_experts": 12,
}

# ── Sélection spectrale L² (haut du spectre) ───────────────────────────────
# On prend les plus grandes valeurs propres de G0 = (1/n) AᵀA.
# Contrôles indépendants :
#   max_n_var   : nombre maximal de vecteurs
#   mu_ceil     : seuil plafond relatif (μ / μ_max)  — on garde μ >= mu_ceil * μ_max
#   mu_floor    : seuil plancher relatif (μ / μ_max) — on garde μ >= mu_floor * μ_max
#                 (en pratique mu_floor sert de coupe basse ; mu_ceil permet une bande)
params_L2 = {
    "max_n_var": 600,
    "mu_ceil": 1e6,     # 1.0 = pas de plafond (on prend jusqu'au max)
    "mu_floor": 1e-6,   # coupe relative basse
}

# ── Sélection spectrale Sobolev (bas du spectre de H_K) ────────────────────
# On prend les plus petites valeurs propres de H_K = Vᵀ G_H V.
# Contrôles indépendants :
#   max_n_sob   : nombre maximal de vecteurs
#   lam_ceil    : seuil plafond (valeurs absolues ou relatives selon use_relative)
#   lam_floor   : seuil plancher
#   use_relative: si True, les seuils sont relatifs à λ_max de H_K
params_Sob = {
    "max_n_sob": 400,
    "lam_ceil": 1e1,        # on garde λ <= lam_ceil * (λ_max si relative)
    "lam_floor": 1e-9,       # on garde λ >= lam_floor * ...
    "use_relative": False,  # seuils absolus par défaut (plus intuitif pour le bas du spectre)
}


# ══════════════════════════════════════════════════════════════════════════════
# DONNÉES : MNIST, chiffres neg_digits vs pos_digits, réduits 28×28 -> img_size²
# (chargement via sklearn/openml pour éviter la dépendance TensorFlow)
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.datasets import fetch_openml

def load_mnist_binary(neg_list, pos_list, img_size, n_train, n_test, n_unlabeled, seed):
    mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
    X = mnist.data.astype(float).reshape(-1, 28, 28)
    yraw = mnist.target.astype(int)

    mask = np.isin(yraw, neg_list + pos_list)
    X = X[mask]; yraw = yraw[mask]
    y = np.where(np.isin(yraw, pos_list), 1., -1.)

    b = 28 // img_size
    X = X.reshape(-1, img_size, b, img_size, b).mean(axis=(2, 4))
    X = X.reshape(len(X), img_size * img_size)
    X = (X - X.mean(0)) / (X.std(0) + 1e-8)

    rng = np.random.default_rng(seed)
    neg = np.where(y < 0)[0]; pos = np.where(y > 0)[0]
    rng.shuffle(neg); rng.shuffle(pos)

    ntr = n_train // 2; nte = n_test // 2
    itr = np.concatenate([neg[:ntr], pos[:ntr]])
    ite = np.concatenate([neg[ntr:ntr+nte], pos[ntr:ntr+nte]])
    used = set(itr) | set(ite)
    pool = np.array([i for i in range(len(X)) if i not in used])
    iu = rng.choice(pool, n_unlabeled, replace=False)

    rng.shuffle(itr); rng.shuffle(ite); rng.shuffle(iu)
    return X[itr], y[itr], X[ite], y[ite], X[iu]

X_train, y_train, X_test, y_test, X_unlabeled = load_mnist_binary(
    params_shared["neg_digits"], params_shared["pos_digits"], params_shared["img_size"],
    params_shared["n_train"], params_shared["n_test"], params_shared["n_unlabeled"],
    seed=params_shared["seed"])

X_all = np.vstack([X_train, X_unlabeled]); N_all = len(X_all)
d = X_train.shape[1]

print(f"digits {params_shared['neg_digits']} vs {params_shared['pos_digits']}, "
      f"{params_shared['img_size']}×{params_shared['img_size']} -> dim {d}")
print(f"  train: {len(X_train)}  test: {len(X_test)}  unlabeled: {len(X_unlabeled)}")
print(f"  train (+1) = {(y_train>0).sum()}/{len(y_train)}")
print(f"  test  (+1) = {(y_test>0).sum()}/{len(y_test)}")
_bal_te = (y_test > 0).mean()
if _bal_te < 0.3 or _bal_te > 0.7:
    print(f"  [!] classes déséquilibrées ({_bal_te:.1%} de +1) — préférer l'AUC.")


def sample_candidates_aniso(X_train, X_unlabeled, n_candidates, train_ratio, rng, dd,
                            sigma_min, sigma_max):
    n_tr = min(int(round(n_candidates * train_ratio)), X_train.shape[0])
    n_ul = min(n_candidates - n_tr, X_unlabeled.shape[0]); parts = []
    if n_tr > 0:
        parts.append(X_train[rng.choice(X_train.shape[0], n_tr, replace=False)])
    if n_ul > 0:
        parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0], n_ul, replace=False)])
    centers = np.vstack(parts)
    log_s = rng.uniform(np.log(sigma_min), np.log(sigma_max), size=(len(centers), dd))
    sigmas = np.exp(log_s)
    return centers, sigmas


# ══════════════════════════════════════════════════════════════════════════════
# GAUSSIENNES ANISOTROPES DIRECTIONNELLES
# phi(x)=exp(-Σ(x_i-c_i)²/(2σ_i²)).
# ══════════════════════════════════════════════════════════════════════════════
def gaussian_features_aniso(X, centers, sigmas):
    diff = X[:, None, :] - centers[None, :, :]
    sq = np.sum(diff**2 / sigmas[None, :, :]**2, axis=2)
    return np.exp(-sq / 2)


def build_G_gauss_aniso_directional_streaming(X_cloud, centers, sigmas, weights, n_dirs,
                                              batch_dirs=10, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    n, d_ = X_cloud.shape
    m = centers.shape[0]
    diff = X_cloud[:, None, :] - centers[None, :, :]
    inv2 = 1.0 / sigmas**2
    sq = np.sum(diff**2 * inv2[None, :, :], axis=2)
    phi = np.exp(-sq / 2)

    w0 = weights.get(0, 0.); w1 = weights.get(1, 0.)
    w2 = weights.get(2, 0.); w3 = weights.get(3, 0.)
    need2 = w2 != 0; need3 = w3 != 0

    G = np.zeros((m, m))
    if w0:
        G += w0 * (phi.T @ phi) / n
    if need2:
        TR = (-np.sum(inv2, axis=1)[None, :] + np.sum(diff**2 * inv2[None, :, :]**2, axis=2)) * phi
        TRG = (TR.T @ TR) / n
    if need3:
        Csum = -np.sum(inv2, axis=1)
        Dq = np.sum(diff**2 * inv2[None, :, :]**2, axis=2)
        coefV = 2 * inv2[None, :, :]**2 - (Csum[None, :, None] + Dq[:, :, None]) * inv2[None, :, :]
        V = phi[:, :, None] * diff * coefV
        VVG = np.einsum('nid,njd->ij', V, V) / n

    MC1_sum = np.zeros((m, m)); MC2_sum = np.zeros((m, m)); MC3_sum = np.zeros((m, m))
    done = 0
    while done < n_dirs:
        K = min(batch_dirs, n_dirs - done)
        U = rng.normal(size=(n, K, d_))
        U /= np.linalg.norm(U, axis=2, keepdims=True)
        A = np.einsum('nmd,nkd,md->nmk', diff, U, inv2)
        B = np.einsum('nkd,md->nmk', U**2, inv2)
        gp = -A; gpp = -B
        if w1:
            D1 = gp * phi[:, :, None]
            D1f = D1.transpose(0, 2, 1).reshape(-1, m)
            MC1_sum += D1f.T @ D1f
            del D1, D1f
        if need2 or need3:
            D2 = (gpp + gp**2) * phi[:, :, None]
            if need2:
                D2f = D2.transpose(0, 2, 1).reshape(-1, m)
                MC2_sum += D2f.T @ D2f
                del D2f
            if need3:
                D3 = (3 * gp * gpp + gp**3) * phi[:, :, None]
                D3f = D3.transpose(0, 2, 1).reshape(-1, m)
                MC3_sum += D3f.T @ D3f
                del D3, D3f
            del D2
        del U, A, B, gp, gpp
        done += K

    if w1:
        G += w1 * d_ * MC1_sum / (n * n_dirs)
    if need2:
        MC2 = MC2_sum / (n * n_dirs)
        G += w2 * (d_ * (d_ + 2) * MC2 - TRG) / 2
    if need3:
        MC3 = MC3_sum / (n * n_dirs)
        G += w3 * (d_ * (d_ + 2) * (d_ + 4) * MC3 - 9 * VVG) / 6
    return G


# ══════════════════════════════════════════════════════════════════════════════
# Sélection spectrale flexible
# ══════════════════════════════════════════════════════════════════════════════
def select_top_spectrum(eigvals, eigvecs, max_n, ceil_rel, floor_rel):
    """
    Sélectionne les plus grandes valeurs propres (haut du spectre).
    eigvals sont supposés triés croissants (np.linalg.eigh).
    Retourne mask booléen sur les indices originaux.
    """
    m = len(eigvals)
    # ordre décroissant
    order = np.argsort(eigvals)[::-1]
    mu = eigvals[order]
    mu_max = mu[0] if mu[0] > 0 else 1.0
    # bande relative
    keep = (mu >= floor_rel * mu_max) & (mu <= ceil_rel * mu_max)
    # limiter au max_n premiers qui passent le filtre
    selected = []
    for i, ok in enumerate(keep):
        if ok:
            selected.append(order[i])
            if len(selected) >= max_n:
                break
    mask = np.zeros(m, dtype=bool)
    mask[selected] = True
    return mask, eigvals[mask], eigvecs[:, mask]


def select_bottom_spectrum(eigvals, eigvecs, max_n, ceil, floor, use_relative=False):
    """
    Sélectionne les plus petites valeurs propres (bas du spectre).
    eigvals triés croissants.
    """
    m = len(eigvals)
    order = np.argsort(eigvals)  # croissant
    lam = eigvals[order]
    scale = lam[-1] if (use_relative and lam[-1] > 0) else 1.0
    keep = (lam >= floor * scale) & (lam <= ceil * scale)
    selected = []
    for i, ok in enumerate(keep):
        if ok:
            selected.append(order[i])
            if len(selected) >= max_n:
                break
    mask = np.zeros(m, dtype=bool)
    mask[selected] = True
    return mask, eigvals[mask], eigvecs[:, mask]


# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 : experts gaussiens anisotropes — DOUBLE DIAGONALISATION
#   L² (haut spectre)  →  H^s (bas spectre)  →  Ridge
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nPhase 1 (gaussien anisotrope, double diag L2→Sobolev) : "
      f"{params_gauss['n_experts']} experts indépendants...")
print(f"  params_L2  = {params_L2}")
print(f"  params_Sob = {params_Sob}")

experts = []
F_train_list = []; F_test_list = []
times_p1 = []

for e in range(params_gauss["n_experts"]):
    t0 = time.time()
    rng = np.random.default_rng(seed=100 + e)

    # ── dictionnaire inchangé ──────────────────────────────────────────────
    candidates, sigmas_cand = sample_candidates_aniso(
        X_train, X_unlabeled, params_gauss["n_dict"], params_shared["train_center_ratio"],
        rng, d, params_gauss["sigma_min"], params_gauss["sigma_max"])

    A_cand = gaussian_features_aniso(X_train, candidates, sigmas_cand)
    corr = (A_cand.T @ y_train) / len(X_train)
    scores = corr**2

    k = min(params_gauss["n_centres"], len(candidates))
    top_k = np.argsort(scores)[-k:]
    centers = candidates[top_k]
    sigmas_sel = sigmas_cand[top_k]
    A = A_cand[:, top_k]                    # (n_train, m)
    Atest = gaussian_features_aniso(X_test, centers, sigmas_sel)
    m = A.shape[1]
    n = A.shape[0]

    # ── Étape 1 : Gram L² / variance sur le train ──────────────────────────
    G0 = (A.T @ A) / n
    # centrage optionnel des colonnes (moyenne nulle) — déjà implicitement
    # proche de 0 car les features sont positives mais on ne force pas ici
    eigvals0, eigvecs0 = np.linalg.eigh(G0)   # croissants

    mask_var, mu_sel, V = select_top_spectrum(
        eigvals0, eigvecs0,
        max_n=params_L2["max_n_var"],
        ceil_rel=params_L2["mu_ceil"],
        floor_rel=params_L2["mu_floor"],
    )
    n_var_eff = V.shape[1]
    if n_var_eff == 0:
        raise RuntimeError(f"Expert {e}: aucune direction L2 sélectionnée")

    # V est orthonormale (colonnes) pour le produit scalaire euclidien

    # ── Étape 2 : Gram Sobolev (mécanisme actuel) puis projection ──────────
    n_G = min(N_all, params_gauss["n_G"])
    idx_G = rng.choice(N_all, size=n_G, replace=False)
    G_H = build_G_gauss_aniso_directional_streaming(
        X_all[idx_G], centers, sigmas_sel, params_gauss["weights"],
        n_dirs=params_shared["n_dirs"], batch_dirs=params_shared["batch_dirs"],
        rng=np.random.default_rng(9000 + e))

    H_K = V.T @ G_H @ V                     # (n_var_eff, n_var_eff)
    # symétrisation numérique
    H_K = 0.5 * (H_K + H_K.T)

    eigvals_H, eigvecs_H = np.linalg.eigh(H_K)  # croissants

    mask_sob, lam_sel, A_sob = select_bottom_spectrum(
        eigvals_H, eigvecs_H,
        max_n=params_Sob["max_n_sob"],
        ceil=params_Sob["lam_ceil"],
        floor=params_Sob["lam_floor"],
        use_relative=params_Sob["use_relative"],
    )
    n_sob_eff = A_sob.shape[1]
    if n_sob_eff == 0:
        raise RuntimeError(f"Expert {e}: aucune direction Sobolev sélectionnée")

    # ── fonctions finales w_i = V a_i  (colonnes de W) ─────────────────────
    W = V @ A_sob                           # (m, n_sob_eff)
    # normalisation euclidienne déjà |a_i|=1 et V ortho ⇒ |w_i|_2 ≈ 1

    # ── Étape 3 : évaluation + Ridge classique dans la base des w_i ────────
    F_tr = A @ W                            # (n_train, n_sob_eff)
    F_te = Atest @ W

    # Ridge avec régularisation simple (λ I)
    # On résout (FᵀF/n + λ I) α = Fᵀy / n
    FtF = (F_tr.T @ F_tr) / n
    rhs = (F_tr.T @ y_train) / n
    M_ridge = FtF + params_gauss["lambda_reg"] * np.eye(n_sob_eff)
    try:
        alpha = np.linalg.solve(M_ridge, rhs)
    except np.linalg.LinAlgError:
        alpha = np.linalg.lstsq(M_ridge, rhs, rcond=None)[0]

    f_tr = F_tr @ alpha
    f_te = F_te @ alpha
    loss_data = np.mean((y_train - f_tr)**2)
    # énergie Sobolev approximative des coefficients dans la base w
    # (optionnel, pour info)
    loss_reg = params_gauss["lambda_reg"] * float(alpha @ alpha)

    t1 = time.time()
    times_p1.append(t1 - t0)

    F_train_list.append(f_tr)
    F_test_list.append(f_te)
    experts.append({
        'type': 'gauss',
        'centers': centers,
        'sigmas': sigmas_sel,
        'W': W,
        'alpha': alpha,
        'n_var': n_var_eff,
        'n_sob': n_sob_eff,
        'mu_sel': mu_sel,
        'lam_sel': lam_sel,
    })

    print(f"\n  gauss {e+1}/{params_gauss['n_experts']}")
    print(f"    dict m={m}  |  n_var={n_var_eff} (max={params_L2['max_n_var']})  "
          f"|  n_sob={n_sob_eff} (max={params_Sob['max_n_sob']})")
    print(f"    L2  μ : min_sel={mu_sel.min():.3e}  max_sel={mu_sel.max():.3e}  "
          f"(μ_max global={eigvals0[-1]:.3e})")
    print(f"    Sob λ : min_sel={lam_sel.min():.3e}  max_sel={lam_sel.max():.3e}  "
          f"(λ_min global={eigvals_H[0]:.3e}, λ_max={eigvals_H[-1]:.3e})")
    print(f"    loss={loss_data+loss_reg:.6f} (data={loss_data:.6f} reg≈{loss_reg:.6f}) | "
          f"{cls_str(f_te, y_test)} | {times_p1[-1]:.1f}s")


F_train = np.column_stack(F_train_list)
F_test = np.column_stack(F_test_list)
n_exp = len(experts)
expert_names = [f"gauss{i+1}" for i in range(n_exp)]


# ══════════════════════════════════════════════════════════════════════════════
# Agrégation simple des experts (moyenne des prédictions) — pas de Phase 2
# ══════════════════════════════════════════════════════════════════════════════
pred_mean_train = F_train.mean(axis=1)
pred_mean_test = F_test.mean(axis=1)

print(f"\n  Moyenne des {n_exp} experts : train {cls_str(pred_mean_train, y_train)}  "
      f"test {cls_str(pred_mean_test, y_test)}")


# ══════════════════════════════════════════════════════════════════════════════
# COMPARAISONS : Ridge polynomial, SVM RBF, Ridge RBF
# ══════════════════════════════════════════════════════════════════════════════
print("\nCalibration Ridge polynomial...")
poly = PolynomialFeatures(degree=min(params_shared["deg_P"], 8), include_bias=False)
ridge_poly = Ridge(alpha=1e-8)
ridge_poly.fit(poly.fit_transform(X_train), y_train)
pred_ridgepoly_te = ridge_poly.predict(poly.transform(X_test))

print("Calibration SVM RBF (GridSearchCV)...")
param_grid_svm = {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.01, 0.1, 1]}
svm = GridSearchCV(SVC(kernel="rbf"), param_grid_svm, cv=5, n_jobs=-1)
svm.fit(X_train, y_train)
pred_svm_te = svm.decision_function(X_test)

print("Calibration Ridge RBF (GridSearchCV)...")
param_grid_kr = {"alpha": [1e-3, 1e-2, 1e-1, 1.0], "gamma": [0.001, 0.01, 0.1, 1]}
kr = GridSearchCV(KernelRidge(kernel="rbf"), param_grid_kr, cv=5, n_jobs=-1)
kr.fit(X_train, y_train)
pred_kr_te = kr.predict(X_test)

print("\n" + "=" * 80)
print("RÉSUMÉ")
print("=" * 80)
for i in range(n_exp):
    print(f"Expert {expert_names[i]} (n_var={experts[i]['n_var']}, n_sob={experts[i]['n_sob']}) : "
          f"{cls_str(F_test[:, i], y_test)}")
print(f"Moyenne des experts                                      : {cls_str(pred_mean_test, y_test)}")
print(f"Ridge polynomial                                         : {cls_str(pred_ridgepoly_te, y_test)}")
print(f"SVM RBF     (best={svm.best_params_})                    : {cls_str(pred_svm_te, y_test)}")
print(f"Ridge RBF   (best={kr.best_params_})                     : {cls_str(pred_kr_te, y_test)}")
print(f"\nn_train={params_shared['n_train']}  n_unlabeled={params_shared['n_unlabeled']}  "
      f"n_test={params_shared['n_test']}  n_dirs={params_shared['n_dirs']}")
print(f"temps phase 1 : {sum(times_p1):.1f}s total ({np.mean(times_p1):.1f}s/expert)")
print("\nDétail par expert :")
for i in range(n_exp):
    print(f"  {expert_names[i]:6s} : {cls_str(F_test[:,i], y_test)}  "
          f"n_var={experts[i]['n_var']} n_sob={experts[i]['n_sob']}")


digits [3, 5] vs [8], 7×7 -> dim 49
  train: 300  test: 5000  unlabeled: 5700
  train (+1) = 150/300
  test  (+1) = 2500/5000

Phase 1 (gaussien anisotrope, double diag L2→Sobolev) : 12 experts indépendants...
  params_L2  = {'max_n_var': 600, 'mu_ceil': 1000000.0, 'mu_floor': 1e-06}
  params_Sob = {'max_n_sob': 400, 'lam_ceil': 10.0, 'lam_floor': 1e-09, 'use_relative': False}

  gauss 1/12
    dict m=800  |  n_var=300 (max=600)  |  n_sob=296 (max=400)
    L2  μ : min_sel=3.987e-05  max_sel=2.599e-01  (μ_max global=2.599e-01)
    Sob λ : min_sel=3.044e-09  max_sel=3.613e-01  (λ_min global=-4.991e-04, λ_max=3.613e-01)
    loss=0.028781 (data=0.012765 reg≈0.016016) | acc=0.8998 AUC=0.9595 | 46.3s
